In [ ]:
def get_temp(epc):
    import requests
    
    lab = "38.7557335&lon=-9.1582073"
    lis = "38.736946&lon=-9.142685"
    sei = "38.625833&lon=-9.086561"
    
    end_point = f"https://api.openweathermap.org/data/3.0/onecall/timemachine?dt={epc}&appid=523ca5a0726f990c05fae636397f252a&units=metric&lat={sei}"

    url = end_point

    response = requests.get(url)

    # Check status code
    if response.status_code == 200:
        # Parse the JSON from the response
        data = response.json()
        return data
    else:
        print(f"Request failed with status code {response.status_code}")

In [ ]:
import pandas as pd
# agg_excel = pd.read_excel('pre_final_aggregated.xlsx')
agg_excel = pd.read_csv('aggregated_data.csv')

In [ ]:
print(set(agg_excel["environment"]))

In [ ]:
import json
import numpy as np

for idx, row in agg_excel.iterrows():
    
    if row["environment"] == "household":
        print("Processing idx: ", idx, row["temperature"], row["humidity"], row["pressure"])
    

In [ ]:
agg_excel.columns

In [ ]:
agg_excel.loc[648, "status"] = 1

In [ ]:
agg_excel["weather"] = 0
agg_excel["temperature"] = 0
agg_excel["humidity"] = 0
agg_excel["pressure"] = 0

In [ ]:
import json
import numpy as np

for idx, row in agg_excel.iterrows():
    
    if row["environment"] == "household" and np.isnan(row["weather"]):

        
        # break
        
        
        file_path = row["file_path"]
        
        csv = pd.read_csv(file_path)

        # try:
        #     get_record_epoch = csv.loc[0]["data_time_stamp"]
        # except:
        #     continue
        get_record_epoch = iso_to_epoch(row["timestamp"])
        
        try:
            
            rounded_epoch = round_epoch(get_record_epoch)
            print(rounded_epoch)
            temp_data = get_temp(rounded_epoch)
        except:
            print("Error at ", idx)
            continue
        
        temperature = temp_data['data'][0]['temp']
        humidity = temp_data['data'][0]['humidity']
        pressure = temp_data['data'][0]['pressure']
        
        
        print("Processing idx: ", idx, row["file_path"], row["timestamp"], temperature, humidity, pressure)
        
        save_to_excel(idx, temperature, humidity, pressure)
            

In [ ]:
def save_to_excel(idx, temperature, humidity, pressure):
    agg_excel.loc[idx, "temperature"] = temperature
    agg_excel.loc[idx, "humidity"] = humidity
    agg_excel.loc[idx, "pressure"] = pressure
    agg_excel.loc[idx, "weather"] = 1

    # agg_excel.to_excel('pre_final_aggregated.xlsx', index=False)

In [ ]:
agg_excel.to_csv('aggregated_data.csv', index=False)

In [ ]:
def round_epoch(epoch):
    import datetime
    
    timestamp = epoch
    
    # Step 1: Convert timestamp to hours
    hours = timestamp / 3600

    # Step 2: Round hours to nearest integer
    rounded_hours = round(hours)

    # Step 3: Convert hours back to seconds
    rounded_timestamp = rounded_hours * 3600

    # Step 4: Convert to a datetime (local time)
    rounded_dt = datetime.datetime.fromtimestamp(rounded_timestamp)
    
    return rounded_timestamp

In [ ]:
round_epoch(1671440140.72)

In [ ]:
temp_data = get_temp()

In [ ]:
from datetime import datetime, timezone

def iso_to_epoch(iso_str: str) -> int:
    """
    Convert ISO-8601 timestamp (e.g. '2024-08-19T02:56:00Z')
    to a rounded Unix epoch (seconds).
    """
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    return int(dt.replace(tzinfo=timezone.utc).timestamp())


In [ ]:
ep = iso_to_epoch("2024-08-19T02:56:00Z")
temp_data = get_temp(ep)

In [ ]:
temp_data